In [83]:
import pandas as pd

df = pd.read_csv("case3_hospital_appointments_raw.csv")

#                     Part A - Initial Inspection

# 1. Load the CSV into df. Display the first 10 rows, shape, columns, dtypes, and info().
#                   columnss to clean
# Age, Appointment_Date, Fee ,Insurance, Rating

print(df.head(10))
print(df.shape) #rows
print(df.columns) #columns
print(df.info())
print(df.dtypes)


# 2. Identify at least five data-quality or data-type problems visible in the file.

# problem 1 -> Age = thirty eight
# problem 2 -> Age = NaN
# problem 3 -> Fee = 1,500
# probelm 4 -> Appointment Date = mixed date formats(13-08-2026 & 13/08/2026) also bad date
# Problem 5 -> Insurance  = (Y, N, TRUE, FALSE)
# Problem 6 -> Rating = Not Rated
# Problem 7 -> Appointment_ID  = A010 and A003 are duplicated

# 3. Create a working copy named df_clean.
df_clean = df.copy()
df_clean


  Appointment_ID   Patient_Name   Department       City           Age  \
0           A001   Meera Patel    Cardiology   Vadodara            45   
1           A002     Rohan Shah  Orthopedics  Ahmedabad  thirty eight   
2           A003     Diya Mehta  Dermatology      Surat            29   
3           A004    Aarav Desai   Cardiology   Vadodara           NaN   
4           A005    Priya Joshi          ENT  Ahmedabad            34   
5           A006      Arjun Rao  Orthopedics      Surat            52   
6           A007   Neha Trivedi  Dermatology   Vadodara            41   
7           A008     Vivek Iyer   Cardiology  Ahmedabad            60   
8           A009     Kavya Nair          ENT      Surat            27   
9           A010    Rahul Verma  Orthopedics   Vadodara           48    

  Appointment_Date   Time    Fee Payment_Status Insurance     Rating  
0       12-08-2026  09:30   1200           Paid       Yes        4.8  
1       12/08/2026  10:15   1500        Pending       

,Appointment_ID,Patient_Name,Department,City,Age,Appointment_Date,Time,Fee,Payment_Status,Insurance,Rating
0,A001,Meera Patel,Cardiology,Vadodara,45,12-08-2026,09:30,1200,Paid,Yes,4.8
1,A002,Rohan Shah,Orthopedics,Ahmedabad,thirty eight,12/08/2026,10:15,1500,Pending,No,4.2
2,A003,Diya Mehta,Dermatology,Surat,29,13-08-2026,11:00,900,Paid,Y,4.6
3,A004,Aarav Desai,Cardiology,Vadodara,NaN,13/08/2026,12:30,1200,Paid,N,4.5
4,A005,Priya Joshi,ENT,Ahmedabad,34,bad date,14:00,800,Pending,Yes,Not Rated
5,A006,Arjun Rao,Orthopedics,Surat,52,14-08-2026,09:00,"1,500",Paid,No,4.7
6,A007,Neha Trivedi,Dermatology,Vadodara,41,14/08/2026,10:45,NaN,Paid,TRUE,4.3
7,A008,Vivek Iyer,Cardiology,Ahmedabad,60,15-08-2026,11:30,1200,Cancelled,FALSE,3.9
8,A009,Kavya Nair,ENT,Surat,27,15/08/2026,13:00,800,Paid,Yes,4.9
9,A010,Rahul Verma,Orthopedics,Vadodara,48,16-08-2026,09:45,1500,Pending,No,4.1


In [81]:
# Part B - Chapter 5: Data Type Conversion
# 1. Remove leading/trailing spaces from Patient_Name.

df_clean["Patient_Name"] = df_clean["Patient_Name"].str.strip()
# for i in df_clean["Patient_Name"]:
#   print(repr(i))

# 2. Convert Age safely to a nullable integer. Invalid age text must become missing.
df_clean["Age"] = pd.to_numeric(df_clean["Age"], errors='coerce')
df_clean["Age"] = df_clean["Age"].astype("Int64")
df_clean["Age"]

# 3. Clean Fee values containing commas/spaces and convert Fee to numeric.
df_clean["Fee"] = df_clean["Fee"].str.strip()
df_clean["Fee"] = df_clean["Fee"].str.replace(",", "")
df_clean["Fee"] = pd.to_numeric(df_clean["Fee"], errors='coerce')
df_clean["Fee"]
# 4. Convert Appointment_Date to datetime using the mixed day-first formats. Invalid dates should become NaT.
df_clean["Appointment_Date"] = pd.to_datetime(df_clean["Appointment_Date"], errors='coerce', format="mixed", dayfirst=True)
df_clean["Appointment_Date"]

# 5. Standardize Insurance values (Yes/Y/TRUE and No/N/FALSE) and convert them to nullable Boolean.
df_clean["Insurance"] = df_clean["Insurance"].replace({
  "Y": "Yes", "N": "No", "TRUE":"Yes", "FALSE": "No"
}).replace({
  "Yes": True, "No": False
}).astype("boolean")

df_clean["Insurance"]

# 6. Convert Rating to numeric safely.
df_clean["Rating"] = pd.to_numeric(df_clean["Rating"], errors='coerce')
df_clean["Rating"]

# 7. Convert Department, City, and Payment_Status to category dtype.
df_clean["Department"] = df_clean["Department"].astype("category")
df_clean["City"] = df_clean["City"].astype("category")
df_clean["Payment_Status"] = df_clean["Payment_Status"].astype("category")
# 8. Display the final dtypes after conversion and identify all conversion-created missing values.
df_clean.dtypes


C:\Users\KARAN\AppData\Local\Temp\ipykernel_23960\2480250307.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  }).replace({


Appointment_ID              object
Patient_Name                object
Department                category
City                      category
Age                          Int64
Appointment_Date    datetime64[ns]
Time                        object
Fee                        float64
Payment_Status            category
Insurance                  boolean
Rating                     float64
dtype: object

In [82]:
# Part C - Chapter 3: Missing Values
# 1. Count missing values after conversion.
# 2. Display every row containing at least one missing value.
# 3. Fill missing Age with the median Age.
# 4. Fill missing Fee using the median Fee of the corresponding Department.
# 5. Fill missing Rating using the median Rating of the corresponding Department.
# 6. The appointment with an invalid Appointment_Date cannot be used for date-based scheduling analysis. Remove that row.
# 7. Verify that the remaining dataset has no missing values in Age, Fee, Rating, or Appointment_Date.

